# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we print out the available record sets in the dataset along with their `@id`s. Then we inspect fields and columns from one record set as an example.

In [ ]:
# List all available record sets and their @id
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")

if len(record_sets) == 0:
    print("No record sets found in the metadata.")
else:
    # List fields and columns for the first record set
    example_rs = record_sets[0]
    print(f"\nFields in RecordSet '@id': {example_rs.id}:")
    for field in example_rs.fields:
        print(f"  - Field: {field.name}, @id: {field.id}")
        if hasattr(field, 'columns'):
            for column in getattr(field, 'columns', []):
                print(f"    - Column: {column.name}, @id: {column.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We extract all available record sets into pandas DataFrames using their `@id`. The column names in the DataFrame correspond to field or column `@id`s.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"  No records found for RecordSet @id: {record_set_id}")
        dataframes[record_set_id] = pd.DataFrame()

# For demonstration, select the first non-empty DataFrame
example_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        example_record_set_id = rs_id
        break

if example_record_set_id:
    print(f"\nColumns in DataFrame (RecordSet @id: {example_record_set_id}):")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No data loaded to DataFrame from any RecordSet. Check dataset schema for available records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a numeric field (by `@id`) for filtering and normalization. Adjust the variable assignments as needed based on the previous cells.

In [ ]:
import numpy as np

# Choose a record set and fields for demonstration (update these as per your schema)
record_set_id = example_record_set_id

if record_set_id is not None and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]
    
    # Try to guess a numeric field by checking dtypes
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        # Fallback: Use the first column
        numeric_field_id = df.columns[0]
    
    print(f"Using numeric field (by @id): {numeric_field_id}")
    
    # Filtering records (example: value greater than threshold)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalization
    filtered_df = filtered_df.copy()
    col_normalized = f"{numeric_field_id}_normalized"
    filtered_df[col_normalized] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_normalized]].head())
    
    # Grouping if categorical field exists
    possible_group_fields = [col for col in df.columns if df[col].dtype=='object']
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        print(f"Grouping by {group_field_id}:")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean").reset_index()
        display(grouped_df.head())
    else:
        print("No categorical/group field found to group by.")
else:
    print("No valid DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset below. Adjust column names and field `@id`s as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]
    # Histogram of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id + ' (@id)')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    # If a group field exists, boxplot
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id + ' (@id)')
        plt.ylabel(numeric_field_id + ' (@id)')
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load and explore a Croissant-based dataset using the `mlcroissant` library.
- All dataset entities (record sets, fields, columns) were referenced by their `@id`.
- You can continue exploring this dataset by examining additional record sets, exploring new relationships, and applying more advanced analyses or visualizations.